# Prueba de credenciales de Gemini

Este notebook carga `GEMINI_API_KEY` y `GEMINI_MODEL` desde `.env` sin mostrar la clave. Primero valida la autenticación y después realiza una inferencia mínima.

> Ejecuta el notebook desde `candidate-solution` y selecciona el entorno de Python del proyecto como kernel.

In [2]:
from pathlib import Path

from google import genai
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class GeminiSettings(BaseSettings):
    model_config = SettingsConfigDict(extra="ignore", case_sensitive=False)

    gemini_api_key: SecretStr
    gemini_model: str = "gemini-3.8-flash"


def find_env_file() -> Path:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd() / "candidate-solution" / ".env",
        *(parent / ".env" for parent in Path.cwd().parents),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError("No se encontró el archivo .env del proyecto.")


env_path = find_env_file()
settings = GeminiSettings(_env_file=env_path, _env_file_encoding="utf-8")

print(f"Configuración cargada desde: {env_path}")
print(f"API key configurada: {'sí' if settings.gemini_api_key.get_secret_value() else 'no'}")
print(f"Modelo configurado: {settings.gemini_model}")

Configuración cargada desde: C:\Users\juanf\Downloads\prueba 2026\EVALUACION\EVALUACION\data-ai-eng-evaluation\data-ai-eng-evaluation\candidate-solution\.env
API key configurada: sí
Modelo configurado: gemini-3.8-flash


## 1. Validar autenticación

Esta celda hace una solicitud pequeña para comprobar que la API acepta la credencial. No imprime la clave.

In [4]:
client = genai.Client(api_key=settings.gemini_api_key.get_secret_value())

try:
    first_model = next(iter(client.models.list()))
    print("✅ Credencial válida: Gemini respondió correctamente.")
    print(f"Modelo visible de ejemplo: {first_model.name}")
except Exception as exc:
    print(f"❌ No se pudo autenticar: {type(exc).__name__}: {exc}")
    raise

✅ Credencial válida: Gemini respondió correctamente.
Modelo visible de ejemplo: models/gemini-2.5-flash


## 2. Enviar una pregunta

Modifica el texto de la variable `question` y ejecuta la celda. Se crea un cliente nuevo en cada ejecución y se imprime la respuesta de la API.

In [5]:
import time

question = "¿Cuál es la capital de Colombia?"
max_attempts = 4

client = genai.Client(api_key=settings.gemini_api_key.get_secret_value())

try:
    for attempt in range(1, max_attempts + 1):
        try:
            response = client.models.generate_content(
                model=settings.gemini_model,
                contents=question,
            )
            print(f"Pregunta: {question}\n")
            print(f"Respuesta de Gemini: {response.text}")
            break
        except Exception as exc:
            is_temporary = "503" in str(exc) or "UNAVAILABLE" in str(exc)
            if not is_temporary or attempt == max_attempts:
                print(f"❌ Error de la API: {type(exc).__name__}: {exc}")
                raise

            wait_seconds = 2 ** attempt
            print(
                f"⏳ Modelo saturado. Reintento {attempt}/{max_attempts - 1} "
                f"en {wait_seconds} segundos..."
            )
            time.sleep(wait_seconds)
finally:
    client.close()

❌ Error de la API: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 3.665234279s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'g

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 3.665234279s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flash'}, 